# Step 7: Production Data Pipeline

## Learning Objectives
1. ETL pipeline design patterns
2. Data lake layered architecture (Bronze / Silver / Gold)
3. Parquet-based data loading and partitioning strategies
4. Data quality validation
5. SCD (Slowly Changing Dimension) patterns
6. MinIO (S3-compatible) integration
7. End-to-end pipeline integration exercise

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random
import time
import os
import shutil
import json

spark = SparkSession.builder \
    .appName('Step7-Data-Pipeline') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.sql.shuffle.partitions', '10') \
    .config('spark.sql.warehouse.dir', '/home/jovyan/data/warehouse') \
    .getOrCreate()

# Pipeline directory structure
LAKE = '/home/jovyan/data/lake'
for layer in ['bronze', 'silver', 'gold']:
    path = f'{LAKE}/{layer}'
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

RAW = '/home/jovyan/data/raw'
if os.path.exists(RAW):
    shutil.rmtree(RAW)
os.makedirs(RAW, exist_ok=True)

print(f'Spark version: {spark.version}')
print(f'✅ Spark UI: http://localhost:4040')
print(f'✅ Data Lake: {LAKE}')
print(f'✅ Raw Data:  {RAW}')

Spark version: 3.5.0
✅ Spark UI: http://localhost:4040
✅ Data Lake: /home/jovyan/data/lake
✅ Raw Data:  /home/jovyan/data/raw


---
## 1. Data Lake Layered Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Data Lake Architecture                    │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Raw Sources          Bronze          Silver         Gold   │
│  (JSON, CSV,    →  (as-is raw)  →  (cleansed)  →  (agg'd)  │
│   API, DB)        Parquet conv.    Dedup          Business  │
│                   Schema apply    Type cast      metrics   │
│                   Partitioning    Validate       Reports   │
│                                  Join/Enrich    ML feats  │
│                                                             │
│  Retention: forever   forever       90d+        by use     │
│  Format:   various    Parquet       Parquet      Parquet   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Why use layers?
- **Bronze**: Preserves raw data (enables debugging and reprocessing)
- **Silver**: Trusted, cleansed data (shared across teams)
- **Gold**: Business decision-making (dashboards, reports, ML)

---
## 2. Generating Raw Data (Source Simulation)

E-commerce scenario: orders, products, and customer data

In [2]:
random.seed(42)

# === Customers CSV ===
regions = ['Seoul', 'Gyeonggi', 'Busan', 'Daegu', 'Incheon', 'Gwangju', 'Daejeon', 'Jeju']
tiers = ['Bronze', 'Silver', 'Gold', 'Platinum']

customer_rows = ['customer_id,name,email,region,tier,signup_date']
for i in range(1, 5001):
    signup = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 730))
    row = f'{i},customer_{i},user_{i}@shop.com,{random.choice(regions)},{random.choice(tiers)},{signup.strftime("%Y-%m-%d")}'
    customer_rows.append(row)

with open(f'{RAW}/customers.csv', 'w') as f:
    f.write('\n'.join(customer_rows))

# === Products JSON ===
categories = ['Electronics', 'Apparel', 'Food', 'Books', 'Sports', 'Furniture', 'Cosmetics']
products_list = []
for i in range(1, 301):
    products_list.append({
        'product_id': i,
        'name': f'product_{i}',
        'category': random.choice(categories),
        'price': round(random.uniform(5000, 500000), -2),
        'cost': round(random.uniform(3000, 300000), -2),
        'is_active': random.random() > 0.1
    })

with open(f'{RAW}/products.json', 'w') as f:
    for p in products_list:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

# === Orders JSON (daily files, 3 days) ===
os.makedirs(f'{RAW}/orders', exist_ok=True)
statuses = ['completed', 'completed', 'completed', 'completed', 'cancelled', 'returned']

for day_offset in range(3):
    date = datetime(2025, 6, 1) + timedelta(days=day_offset)
    date_str = date.strftime('%Y-%m-%d')
    orders_list = []
    
    n_orders = random.randint(8000, 12000)
    for i in range(n_orders):
        order_time = date + timedelta(
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )
        order = {
            'order_id': f'ORD-{date_str}-{i:06d}',
            'customer_id': random.randint(1, 5000),
            'product_id': random.randint(1, 300),
            'quantity': random.randint(1, 5),
            'status': random.choice(statuses),
            'order_time': order_time.isoformat(),
            'payment_method': random.choice(['card', 'bank', 'mobile', 'point']),
        }
        # Intentionally inject quality issues
        if random.random() < 0.02:  # 2% null customer_id
            order['customer_id'] = None
        if random.random() < 0.01:  # 1% negative quantity
            order['quantity'] = -1
        if random.random() < 0.005:  # 0.5% duplicate order_id
            order['order_id'] = f'ORD-{date_str}-{max(0,i-1):06d}'
            
        orders_list.append(order)
    
    with open(f'{RAW}/orders/{date_str}.json', 'w') as f:
        for o in orders_list:
            f.write(json.dumps(o, ensure_ascii=False) + '\n')
    
    print(f'  {date_str}: {n_orders:,} orders generated')

print(f'\n✅ Raw data generation complete')
print(f'  Customers: {RAW}/customers.csv (5,000 rows)')
print(f'  Products:  {RAW}/products.json (300 rows)')
print(f'  Orders:    {RAW}/orders/ (3 daily files)')

  2025-06-01: 8,250 orders generated
  2025-06-02: 9,356 orders generated
  2025-06-03: 11,404 orders generated

✅ Raw data generation complete
  Customers: /home/jovyan/data/raw/customers.csv (5,000 rows)
  Products:  /home/jovyan/data/raw/products.json (300 rows)
  Orders:    /home/jovyan/data/raw/orders/ (3 daily files)


---
## 3. Bronze Layer: Raw Data Ingestion

Save the raw data to Parquet with minimal transformation.
- Explicitly apply schema
- Add ingestion timestamp metadata
- Date-based partitioning

In [3]:
# === Bronze: Customers ===
customers_raw = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv(f'{RAW}/customers.csv')

customers_bronze = customers_raw \
    .withColumn('_ingested_at', F.current_timestamp()) \
    .withColumn('_source', F.lit('raw/customers.csv'))

customers_bronze.write \
    .mode('overwrite') \
    .parquet(f'{LAKE}/bronze/customers')

print(f'Bronze customers: {customers_bronze.count():,}')
customers_bronze.show(3)

Bronze customers: 5,000
+-----------+----------+---------------+--------+------+-----------+--------------------+-----------------+
|customer_id|      name|          email|  region|  tier|signup_date|        _ingested_at|          _source|
+-----------+----------+---------------+--------+------+-----------+--------------------+-----------------+
|          1|customer_1|user_1@shop.com|Gyeonggi|Bronze| 2024-10-16|2026-06-03 09:10:...|raw/customers.csv|
|          2|customer_2|user_2@shop.com|   Daegu|Silver| 2023-10-09|2026-06-03 09:10:...|raw/customers.csv|
|          3|customer_3|user_3@shop.com|Gyeonggi|Bronze| 2023-05-23|2026-06-03 09:10:...|raw/customers.csv|
+-----------+----------+---------------+--------+------+-----------+--------------------+-----------------+
only showing top 3 rows



In [4]:
# === Bronze: Products ===
products_raw = spark.read.json(f'{RAW}/products.json')

products_bronze = products_raw \
    .withColumn('_ingested_at', F.current_timestamp()) \
    .withColumn('_source', F.lit('raw/products.json'))

products_bronze.write \
    .mode('overwrite') \
    .parquet(f'{LAKE}/bronze/products')

print(f'Bronze products: {products_bronze.count():,}')
products_bronze.show(3)

Bronze products: 300
+--------+--------+---------+---------+--------+----------+--------------------+-----------------+
|category|    cost|is_active|     name|   price|product_id|        _ingested_at|          _source|
+--------+--------+---------+---------+--------+----------+--------------------+-----------------+
|   Books| 28200.0|     true|product_1| 37800.0|         1|2026-06-03 09:10:...|raw/products.json|
| Apparel| 14900.0|     true|product_2|387600.0|         2|2026-06-03 09:10:...|raw/products.json|
|    Food|242600.0|     true|product_3|234900.0|         3|2026-06-03 09:10:...|raw/products.json|
+--------+--------+---------+---------+--------+----------+--------------------+-----------------+
only showing top 3 rows



In [5]:
# === Bronze: Orders (daily partitioning) ===
order_schema = StructType([
    StructField('order_id', StringType()),
    StructField('customer_id', IntegerType()),
    StructField('product_id', IntegerType()),
    StructField('quantity', IntegerType()),
    StructField('status', StringType()),
    StructField('order_time', TimestampType()),
    StructField('payment_method', StringType()),
])

orders_raw = spark.read \
    .schema(order_schema) \
    .json(f'{RAW}/orders/')

orders_bronze = orders_raw \
    .withColumn('order_date', F.to_date('order_time')) \
    .withColumn('_ingested_at', F.current_timestamp()) \
    .withColumn('_source', F.lit('raw/orders/'))

orders_bronze.write \
    .mode('overwrite') \
    .partitionBy('order_date') \
    .parquet(f'{LAKE}/bronze/orders')

print(f'Bronze orders: {orders_bronze.count():,}')
print(f'\nDaily distribution:')
orders_bronze.groupBy('order_date').count().orderBy('order_date').show()

# Check partition structure
print('Partition directories:')
for d in sorted(os.listdir(f'{LAKE}/bronze/orders')):
    if d.startswith('order_date='):
        n_files = len([f for f in os.listdir(f'{LAKE}/bronze/orders/{d}') if f.endswith('.parquet')])
        print(f'  {d}: {n_files} parquet files')

Bronze orders: 29,010

Daily distribution:
+----------+-----+
|order_date|count|
+----------+-----+
|2025-06-01| 8250|
|2025-06-02| 9356|
|2025-06-03|11404|
+----------+-----+

Partition directories:
  order_date=2025-06-01: 1 parquet files
  order_date=2025-06-02: 1 parquet files
  order_date=2025-06-03: 1 parquet files


---
## 4. Silver Layer: Data Cleansing

- Deduplication
- NULL / anomalous value handling
- Type casting & standardization
- Data quality validation

In [6]:
# === Data quality check function ===
def check_quality(df, name, checks):
    """Run data quality checks and print report"""
    total = df.count()
    print(f'\n{"=" * 50}')
    print(f'  Data Quality Report: {name}')
    print(f'  Total rows: {total:,}')
    print(f'{"=" * 50}')
    
    all_passed = True
    for check_name, condition, threshold in checks:
        bad_count = df.filter(~condition).count()
        bad_pct = (bad_count / total * 100) if total > 0 else 0
        passed = bad_pct <= threshold
        status = '✅ PASS' if passed else '❌ FAIL'
        print(f'  {status} {check_name}: {bad_count:,} bad rows ({bad_pct:.2f}%, threshold {threshold}%)')
        if not passed:
            all_passed = False
    
    return all_passed

In [7]:
# === Silver: Orders cleansing ===
orders_b = spark.read.parquet(f'{LAKE}/bronze/orders')

# Step 1: Quality check (Bronze)
quality_checks = [
    ('order_id NOT NULL', F.col('order_id').isNotNull(), 0),
    ('customer_id NOT NULL', F.col('customer_id').isNotNull(), 3),
    ('quantity > 0', F.col('quantity') > 0, 2),
    ('status valid', F.col('status').isin('completed', 'cancelled', 'returned'), 1),
    ('order_time NOT NULL', F.col('order_time').isNotNull(), 0),
]

check_quality(orders_b, 'Bronze orders', quality_checks)

# Step 2: Cleanse
orders_silver = orders_b \
    .dropDuplicates(['order_id']) \
    .filter(F.col('order_id').isNotNull()) \
    .filter(F.col('customer_id').isNotNull()) \
    .filter(F.col('quantity') > 0) \
    .filter(F.col('status').isin('completed', 'cancelled', 'returned')) \
    .withColumn('order_hour', F.hour('order_time')) \
    .withColumn('order_dow', F.dayofweek('order_time')) \
    .drop('_ingested_at', '_source')

# Step 3: Save
orders_silver.write \
    .mode('overwrite') \
    .partitionBy('order_date') \
    .parquet(f'{LAKE}/silver/orders')

before = orders_b.count()
after = orders_silver.count()
print(f'\nBronze: {before:,} → Silver: {after:,} ({before - after:,} rows removed, {(before-after)/before*100:.1f}%)')

# Re-check quality on Silver
check_quality(orders_silver, 'Silver orders', quality_checks)


  Data Quality Report: Bronze orders
  Total rows: 29,010
  ✅ PASS order_id NOT NULL: 0 bad rows (0.00%, threshold 0%)
  ✅ PASS customer_id NOT NULL: 599 bad rows (2.06%, threshold 3%)
  ✅ PASS quantity > 0: 301 bad rows (1.04%, threshold 2%)
  ✅ PASS status valid: 0 bad rows (0.00%, threshold 1%)
  ✅ PASS order_time NOT NULL: 0 bad rows (0.00%, threshold 0%)

Bronze: 29,010 → Silver: 27,972 (1,038 rows removed, 3.6%)

  Data Quality Report: Silver orders
  Total rows: 27,972
  ✅ PASS order_id NOT NULL: 0 bad rows (0.00%, threshold 0%)
  ✅ PASS customer_id NOT NULL: 0 bad rows (0.00%, threshold 3%)
  ✅ PASS quantity > 0: 0 bad rows (0.00%, threshold 2%)
  ✅ PASS status valid: 0 bad rows (0.00%, threshold 1%)
  ✅ PASS order_time NOT NULL: 0 bad rows (0.00%, threshold 0%)


True

In [8]:
# === Silver: Customers cleansing ===
customers_b = spark.read.parquet(f'{LAKE}/bronze/customers')

customers_silver = customers_b \
    .dropDuplicates(['customer_id']) \
    .filter(F.col('customer_id').isNotNull()) \
    .withColumn('signup_date', F.to_date('signup_date')) \
    .drop('_ingested_at', '_source')

customers_silver.write \
    .mode('overwrite') \
    .parquet(f'{LAKE}/silver/customers')

print(f'Silver customers: {customers_silver.count():,}')

# === Silver: Products cleansing ===
products_b = spark.read.parquet(f'{LAKE}/bronze/products')

products_silver = products_b \
    .dropDuplicates(['product_id']) \
    .filter(F.col('is_active') == True) \
    .withColumn('margin', F.col('price') - F.col('cost')) \
    .withColumn('margin_pct', F.round(F.col('margin') / F.col('price') * 100, 1)) \
    .drop('_ingested_at', '_source')

products_silver.write \
    .mode('overwrite') \
    .parquet(f'{LAKE}/silver/products')

print(f'Silver products: {products_silver.count():,} (inactive excluded)')
products_silver.show(5)

Silver customers: 5,000
Silver products: 280 (inactive excluded)
+---------+--------+---------+---------+--------+----------+--------+----------+
| category|    cost|is_active|     name|   price|product_id|  margin|margin_pct|
+---------+--------+---------+---------+--------+----------+--------+----------+
|    Books| 28200.0|     true|product_1| 37800.0|         1|  9600.0|      25.4|
|  Apparel| 14900.0|     true|product_2|387600.0|         2|372700.0|      96.2|
|     Food|242600.0|     true|product_3|234900.0|         3| -7700.0|      -3.3|
|     Food|182800.0|     true|product_4|419000.0|         4|236200.0|      56.4|
|Cosmetics|154900.0|     true|product_5|474500.0|         5|319600.0|      67.4|
+---------+--------+---------+---------+--------+----------+--------+----------+
only showing top 5 rows



---
## 5. Gold Layer: Business Metrics

Build aggregation tables for direct use by analytics teams and management.

In [9]:
# Load Silver data
orders_s = spark.read.parquet(f'{LAKE}/silver/orders')
customers_s = spark.read.parquet(f'{LAKE}/silver/customers')
products_s = spark.read.parquet(f'{LAKE}/silver/products')

# === Gold 1: Daily Revenue Summary ===
daily_sales = orders_s \
    .filter(F.col('status') == 'completed') \
    .join(F.broadcast(products_s.select('product_id', 'price', 'category', 'margin')), 'product_id') \
    .groupBy('order_date') \
    .agg(
        F.count('*').alias('order_count'),
        F.countDistinct('customer_id').alias('unique_customers'),
        F.sum(F.col('price') * F.col('quantity')).alias('total_revenue'),
        F.sum(F.col('margin') * F.col('quantity')).alias('total_margin'),
        F.avg(F.col('price') * F.col('quantity')).alias('avg_order_value'),
        F.sum('quantity').alias('total_items'),
    ) \
    .withColumn('margin_rate', F.round(F.col('total_margin') / F.col('total_revenue') * 100, 1)) \
    .orderBy('order_date')

daily_sales.write.mode('overwrite').parquet(f'{LAKE}/gold/daily_sales')

print('=== Gold: Daily Sales Summary ===')
daily_sales.show(truncate=False)

=== Gold: Daily Sales Summary ===
+----------+-----------+----------------+-------------+------------+-----------------+-----------+-----------+
|order_date|order_count|unique_customers|total_revenue|total_margin|avg_order_value  |total_items|margin_rate|
+----------+-----------+----------------+-------------+------------+-----------------+-----------+-----------+
|2025-06-01|5011       |3121            |3.7735279E9  |1.5115894E9 |753048.8724805427|14911      |40.1       |
|2025-06-02|5587       |3379            |4.2420866E9  |1.7060173E9 |759278.0741005907|16700      |40.2       |
|2025-06-03|6839       |3705            |5.273174E9   |2.0679095E9 |771044.597163328 |20781      |39.2       |
+----------+-----------+----------------+-------------+------------+-----------------+-----------+-----------+



In [10]:
# === Gold 2: Category Performance ===
category_perf = orders_s \
    .filter(F.col('status') == 'completed') \
    .join(F.broadcast(products_s.select('product_id', 'price', 'category', 'cost')), 'product_id') \
    .groupBy('category') \
    .agg(
        F.count('*').alias('orders'),
        F.sum(F.col('price') * F.col('quantity')).alias('revenue'),
        F.sum(F.col('cost') * F.col('quantity')).alias('cost'),
        F.countDistinct('customer_id').alias('buyers'),
        F.avg('quantity').alias('avg_qty'),
    ) \
    .withColumn('profit', F.col('revenue') - F.col('cost')) \
    .withColumn('profit_margin_pct', F.round(F.col('profit') / F.col('revenue') * 100, 1)) \
    .orderBy(F.col('revenue').desc())

category_perf.write.mode('overwrite').parquet(f'{LAKE}/gold/category_performance')

print('=== Gold: Category Performance ===')
category_perf.show(truncate=False)

=== Gold: Category Performance ===
+-----------+------+-----------+-----------+------+------------------+-----------+-----------------+
|category   |orders|revenue    |cost       |buyers|avg_qty           |profit     |profit_margin_pct|
+-----------+------+-----------+-----------+------+------------------+-----------+-----------------+
|Sports     |2794  |2.1274984E9|1.4379191E9|2130  |3.0171796707229777|6.895793E8 |32.4             |
|Books      |2402  |2.0840564E9|1.052271E9 |1919  |2.9883430474604498|1.0317854E9|49.5             |
|Furniture  |2561  |2.0233603E9|1.0977113E9|2036  |2.9675907848496683|9.25649E8  |45.7             |
|Apparel    |2489  |1.9756506E9|1.0863748E9|1980  |2.989152269987947 |8.892758E8 |45.0             |
|Food       |2439  |1.8849476E9|1.1118909E9|1916  |3.03280032800328  |7.730567E8 |41.0             |
|Cosmetics  |2619  |1.6076484E9|1.3341049E9|2050  |3.016418480336006 |2.735435E8 |17.0             |
|Electronics|2133  |1.5856268E9|8.830003E8 |1733  |3.022

In [11]:
# === Gold 3: Customer Segments (RFM Analysis) ===
reference_date = datetime(2025, 6, 4)

rfm = orders_s \
    .filter(F.col('status') == 'completed') \
    .join(F.broadcast(products_s.select('product_id', 'price')), 'product_id') \
    .groupBy('customer_id') \
    .agg(
        F.datediff(F.lit(reference_date), F.max('order_time')).alias('recency'),
        F.count('*').alias('frequency'),
        F.sum(F.col('price') * F.col('quantity')).alias('monetary'),
    )

# RFM scores (split each metric into quintiles)
rfm_scored = rfm \
    .withColumn('r_score', F.when(F.col('recency') <= 1, 5)
                           .when(F.col('recency') <= 2, 4)
                           .when(F.col('recency') <= 3, 3)
                           .otherwise(2)) \
    .withColumn('f_score', F.when(F.col('frequency') >= 10, 5)
                           .when(F.col('frequency') >= 7, 4)
                           .when(F.col('frequency') >= 4, 3)
                           .when(F.col('frequency') >= 2, 2)
                           .otherwise(1)) \
    .withColumn('m_score', F.ntile(5).over(
        __import__('pyspark.sql.window', fromlist=['Window']).Window.orderBy('monetary')
    ))

# Segment classification
rfm_final = rfm_scored \
    .withColumn('rfm_sum', F.col('r_score') + F.col('f_score') + F.col('m_score')) \
    .withColumn('segment',
        F.when(F.col('rfm_sum') >= 13, 'Champions')
         .when(F.col('rfm_sum') >= 10, 'Loyal')
         .when(F.col('rfm_sum') >= 7, 'Potential')
         .when(F.col('rfm_sum') >= 4, 'At Risk')
         .otherwise('Lost')
    ) \
    .join(customers_s.select('customer_id', 'name', 'region', 'tier'), 'customer_id')

rfm_final.write.mode('overwrite').parquet(f'{LAKE}/gold/customer_segments')

print('=== Gold: Customer Segments ===')
rfm_final.groupBy('segment').agg(
    F.count('*').alias('customers'),
    F.round(F.avg('monetary'), 0).alias('avg_monetary'),
    F.round(F.avg('frequency'), 1).alias('avg_frequency'),
).orderBy(F.col('avg_monetary').desc()).show()

=== Gold: Customer Segments ===
+---------+---------+------------+-------------+
|  segment|customers|avg_monetary|avg_frequency|
+---------+---------+------------+-------------+
|Champions|      898|   5478048.0|          6.0|
|    Loyal|     2039|   2963809.0|          3.9|
|Potential|     1612|   1320626.0|          2.3|
|  At Risk|      316|    624825.0|          1.1|
+---------+---------+------------+-------------+



In [12]:
# === Gold 4: Hourly Order Patterns ===
hourly_pattern = orders_s \
    .filter(F.col('status') == 'completed') \
    .groupBy('order_hour') \
    .agg(
        F.count('*').alias('orders'),
        F.countDistinct('customer_id').alias('unique_customers'),
    ) \
    .orderBy('order_hour')

hourly_pattern.write.mode('overwrite').parquet(f'{LAKE}/gold/hourly_pattern')

print('=== Gold: Hourly Order Patterns ===')
hourly_data = hourly_pattern.collect()
max_orders = max(row['orders'] for row in hourly_data)
for row in hourly_data:
    bar = '█' * int(row['orders'] / max_orders * 40)
    print(f'  {row["order_hour"]:>2}:00  {row["orders"]:>5} orders  {bar}')

=== Gold: Hourly Order Patterns ===
   0:00    748 orders  ███████████████████████████████████
   1:00    707 orders  █████████████████████████████████
   2:00    754 orders  ███████████████████████████████████
   3:00    772 orders  ████████████████████████████████████
   4:00    816 orders  ██████████████████████████████████████
   5:00    773 orders  ████████████████████████████████████
   6:00    760 orders  ███████████████████████████████████
   7:00    788 orders  █████████████████████████████████████
   8:00    815 orders  ██████████████████████████████████████
   9:00    846 orders  ████████████████████████████████████████
  10:00    764 orders  ████████████████████████████████████
  11:00    808 orders  ██████████████████████████████████████
  12:00    755 orders  ███████████████████████████████████
  13:00    737 orders  ██████████████████████████████████
  14:00    765 orders  ████████████████████████████████████
  15:00    816 orders  ██████████████████████████████████████


---
## 6. MinIO (S3-Compatible) Integration

In production, data lakes are stored on S3/GCS/ADLS.
MinIO is an S3-compatible object storage suitable for local practice.

In [13]:
# MinIO connection test
# Note: Requires the S3A connector. This environment only shows the configuration.

print('''
=== MinIO / S3 Integration Setup ===

Add these configs when creating the SparkSession:

spark = SparkSession.builder \\
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000') \\
    .config('spark.hadoop.fs.s3a.access.key', 'sparklearn') \\
    .config('spark.hadoop.fs.s3a.secret.key', 'sparklearn123') \\
    .config('spark.hadoop.fs.s3a.path.style.access', 'true') \\
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \\
    .getOrCreate()

Usage:
  # Read
  df = spark.read.parquet('s3a://data-lake/silver/orders/')
  
  # Write
  df.write.parquet('s3a://data-lake/gold/daily_sales/')

Required JARs (spark.jars.packages):
  - org.apache.hadoop:hadoop-aws:3.3.4
  - com.amazonaws:aws-java-sdk-bundle:1.12.262

MinIO Console: http://localhost:9001
  ID: sparklearn / PW: sparklearn123
''')


=== MinIO / S3 Integration Setup ===

Add these configs when creating the SparkSession:

spark = SparkSession.builder \
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000') \
    .config('spark.hadoop.fs.s3a.access.key', 'sparklearn') \
    .config('spark.hadoop.fs.s3a.secret.key', 'sparklearn123') \
    .config('spark.hadoop.fs.s3a.path.style.access', 'true') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .getOrCreate()

Usage:
  # Read
  df = spark.read.parquet('s3a://data-lake/silver/orders/')
  
  # Write
  df.write.parquet('s3a://data-lake/gold/daily_sales/')

Required JARs (spark.jars.packages):
  - org.apache.hadoop:hadoop-aws:3.3.4
  - com.amazonaws:aws-java-sdk-bundle:1.12.262

MinIO Console: http://localhost:9001
  ID: sparklearn / PW: sparklearn123



---
## 7. Incremental Load Pattern

In [14]:
# Incremental processing: reprocess only a specific date (partition-level)
def process_orders_incremental(date_str):
    """Process only one day of orders: Bronze → Silver"""
    start = time.time()
    
    # Read only the target date from Bronze (Partition Pruning)
    orders_day = spark.read.parquet(f'{LAKE}/bronze/orders') \
        .filter(F.col('order_date') == date_str)
    
    # Silver cleansing
    silver_day = orders_day \
        .dropDuplicates(['order_id']) \
        .filter(F.col('customer_id').isNotNull()) \
        .filter(F.col('quantity') > 0) \
        .withColumn('order_hour', F.hour('order_time')) \
        .withColumn('order_dow', F.dayofweek('order_time')) \
        .drop('_ingested_at', '_source')
    
    # Overwrite only that partition
    silver_day.write \
        .mode('overwrite') \
        .parquet(f'{LAKE}/silver/orders_incremental/order_date={date_str}')
    
    elapsed = time.time() - start
    count = silver_day.count()
    print(f'  {date_str}: {count:,} rows processed ({elapsed:.2f}s)')
    return count

print('=== Incremental Processing Simulation ===')
total = 0
for date in ['2025-06-01', '2025-06-02', '2025-06-03']:
    total += process_orders_incremental(date)

print(f'\nTotal processed: {total:,}')
print('''
💡 Benefits of incremental processing:
   - Faster: no need to reprocess the entire dataset
   - If one date has issues, only that partition needs reprocessing
   - Partition Pruning minimizes read cost
   
   In production, this is automated daily/hourly with schedulers
   like Airflow or Dagster.
''')

=== Incremental Processing Simulation ===
  2025-06-01: 7,930 rows processed (0.20s)
  2025-06-02: 9,029 rows processed (0.15s)
  2025-06-03: 11,013 rows processed (0.23s)

Total processed: 27,972

💡 Benefits of incremental processing:
   - Faster: no need to reprocess the entire dataset
   - If one date has issues, only that partition needs reprocessing
   - Partition Pruning minimizes read cost
   
   In production, this is automated daily/hourly with schedulers
   like Airflow or Dagster.



---
## 8. SCD Type 2 Pattern (Slowly Changing Dimension)

In [15]:
# SCD Type 2: Managing customer tier change history

# Current customer data (existing)
existing = spark.createDataFrame([
    (1, 'Alice', 'Silver', '2024-01-01', '9999-12-31', True),
    (2, 'Bob', 'Gold', '2024-03-15', '9999-12-31', True),
    (3, 'Charlie', 'Bronze', '2024-06-01', '9999-12-31', True),
], ['customer_id', 'name', 'tier', 'effective_from', 'effective_to', 'is_current'])

# Incoming data (includes tier changes)
incoming = spark.createDataFrame([
    (1, 'Alice', 'Gold'),      # Silver → Gold upgrade!
    (2, 'Bob', 'Gold'),        # no change
    (3, 'Charlie', 'Silver'),  # Bronze → Silver upgrade!
    (4, 'Diana', 'Bronze'),    # new customer
], ['customer_id', 'name', 'tier'])

today = '2025-06-01'
yesterday = '2025-05-31'

# Detect changes
changes = existing.filter(F.col('is_current') == True) \
    .join(incoming, 'customer_id', 'full_outer')

# Changed records: close the old row
closed = existing.filter(F.col('is_current') == True) \
    .join(incoming.select('customer_id', incoming.tier.alias('new_tier')), 'customer_id') \
    .filter(F.col('tier') != F.col('new_tier')) \
    .withColumn('effective_to', F.lit(yesterday)) \
    .withColumn('is_current', F.lit(False)) \
    .drop('new_tier')

# Changed records: insert new row
new_versions = existing.filter(F.col('is_current') == True) \
    .join(incoming.select('customer_id', incoming.name, incoming.tier.alias('new_tier')), 'customer_id') \
    .filter(F.col('tier') != F.col('new_tier')) \
    .select(
        'customer_id',
        incoming.name,
        F.col('new_tier').alias('tier'),
        F.lit(today).alias('effective_from'),
        F.lit('9999-12-31').alias('effective_to'),
        F.lit(True).alias('is_current')
    )

# Unchanged records
unchanged = existing.filter(F.col('is_current') == True) \
    .join(incoming.select('customer_id', incoming.tier.alias('new_tier')), 'customer_id') \
    .filter(F.col('tier') == F.col('new_tier')) \
    .drop('new_tier')

# Completely new records
brand_new = incoming.join(existing.select('customer_id'), 'customer_id', 'left_anti') \
    .withColumn('effective_from', F.lit(today)) \
    .withColumn('effective_to', F.lit('9999-12-31')) \
    .withColumn('is_current', F.lit(True))

# Historical (already closed) records
historical = existing.filter(F.col('is_current') == False)

# Final merge
result = historical \
    .unionByName(closed) \
    .unionByName(unchanged) \
    .unionByName(new_versions) \
    .unionByName(brand_new) \
    .orderBy('customer_id', 'effective_from')

print('=== SCD Type 2 Result ===')
result.show(truncate=False)

print('''
💡 SCD Type 2:
   - Alice: Silver (~05/31) → Gold (06/01~) history preserved
   - Bob: no change
   - Charlie: Bronze (~05/31) → Silver (06/01~) history preserved
   - Diana: newly added
   
   In production, Delta Lake's MERGE command makes this much simpler.
''')

=== SCD Type 2 Result ===
+-----------+-------+------+--------------+------------+----------+
|customer_id|name   |tier  |effective_from|effective_to|is_current|
+-----------+-------+------+--------------+------------+----------+
|1          |Alice  |Silver|2024-01-01    |2025-05-31  |false     |
|1          |Alice  |Gold  |2025-06-01    |9999-12-31  |true      |
|2          |Bob    |Gold  |2024-03-15    |9999-12-31  |true      |
|3          |Charlie|Bronze|2024-06-01    |2025-05-31  |false     |
|3          |Charlie|Silver|2025-06-01    |9999-12-31  |true      |
|4          |Diana  |Bronze|2025-06-01    |9999-12-31  |true      |
+-----------+-------+------+--------------+------------+----------+


💡 SCD Type 2:
   - Alice: Silver (~05/31) → Gold (06/01~) history preserved
   - Bob: no change
   - Charlie: Bronze (~05/31) → Silver (06/01~) history preserved
   - Diana: newly added
   
   In production, Delta Lake's MERGE command makes this much simpler.



---
## 9. Full Pipeline Structure Overview

In [16]:
# Print the data lake structure
print('=== Data Lake Structure ===')
print()
for layer in ['bronze', 'silver', 'gold']:
    layer_path = f'{LAKE}/{layer}'
    if not os.path.exists(layer_path):
        continue
    print(f'📂 {layer.upper()}/')
    for table in sorted(os.listdir(layer_path)):
        table_path = f'{layer_path}/{table}'
        if os.path.isdir(table_path):
            try:
                df = spark.read.parquet(table_path)
                count = df.count()
                cols = len(df.columns)
                print(f'  └─ {table}: {count:,} rows, {cols} columns')
            except:
                print(f'  └─ {table}: (read failed)')
    print()

=== Data Lake Structure ===

📂 BRONZE/
  └─ customers: 5,000 rows, 8 columns
  └─ orders: 29,010 rows, 10 columns
  └─ products: 300 rows, 8 columns

📂 SILVER/
  └─ customers: 5,000 rows, 6 columns
  └─ orders: 27,972 rows, 10 columns
  └─ orders_incremental: 27,972 rows, 10 columns
  └─ products: 280 rows, 8 columns

📂 GOLD/
  └─ category_performance: 7 rows, 8 columns
  └─ customer_segments: 4,865 rows, 12 columns
  └─ daily_sales: 3 rows, 8 columns
  └─ hourly_pattern: 24 rows, 3 columns



In [17]:
print('''
=== Production Data Pipeline Architecture ===

┌────────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
│  Sources   │     │  Bronze  │     │  Silver  │     │   Gold   │
│            │     │          │     │          │     │          │
│ ◦ API      │────→│ raw      │────→│ cleansed │────→│ aggreg.  │
│ ◦ DB CDC   │     │ Parquet  │     │ dedup    │     │ RFM      │
│ ◦ Files    │     │ partition│     │ validate │     │ revenue  │
│ ◦ Kafka    │     │ metadata │     │ join     │     │ reports  │
└────────────┘     └──────────┘     └──────────┘     └──────────┘
                        │                │                │
                   ┌────▼────────────────▼────────────────▼────┐
                   │            Storage (S3/MinIO)             │
                   │         Parquet + Hive Metastore          │
                   └──────────────────────────────────────────┘
                        │                │                │
                   ┌────▼───┐    ┌───────▼──┐    ┌───────▼──┐
                   │Airflow │    │Spark SQL │    │Dashboard │
                   │schedul.│    │  queries │    │BI tools  │
                   └────────┘    └──────────┘    └──────────┘

Production tech stack:
  Storage:    S3 / GCS / ADLS
  Table fmt:  Delta Lake / Apache Iceberg / Hudi
  Processing: Spark (EMR / Dataproc / self-managed)
  Scheduler:  Airflow / Dagster / Prefect
  Catalog:    Hive Metastore / AWS Glue / Data Catalog
  Monitoring: Datadog / Grafana / Spark UI
''')


=== Production Data Pipeline Architecture ===

┌────────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
│  Sources   │     │  Bronze  │     │  Silver  │     │   Gold   │
│            │     │          │     │          │     │          │
│ ◦ API      │────→│ raw      │────→│ cleansed │────→│ aggreg.  │
│ ◦ DB CDC   │     │ Parquet  │     │ dedup    │     │ RFM      │
│ ◦ Files    │     │ partition│     │ validate │     │ revenue  │
│ ◦ Kafka    │     │ metadata │     │ join     │     │ reports  │
└────────────┘     └──────────┘     └──────────┘     └──────────┘
                        │                │                │
                   ┌────▼────────────────▼────────────────▼────┐
                   │            Storage (S3/MinIO)             │
                   │         Parquet + Hive Metastore          │
                   └──────────────────────────────────────────┘
                        │                │                │
                   ┌────▼───┐    ┌───────▼──┐

---
## 📝 Key Takeaways

| Concept | Description |
|---------|-------------|
| **Bronze** | Preserve raw data, minimal transformation, Parquet + metadata |
| **Silver** | Cleansed (dedup, NULL handling, validation, type standardization) |
| **Gold** | Business metrics (revenue, RFM, segments, reports) |
| **Data Quality** | Rule-based validation, threshold setting, automation |
| **Incremental** | Partition-level incremental processing, easy reprocessing |
| **SCD Type 2** | Preserve dimension change history (effective_from/to) |
| **Partition Pruning** | Date partitioning → read only required partitions |

### Full Learning Roadmap Complete! 🎉

| Step | Topic | Key Concepts |
|------|-------|--------------|
| 1 | RDD & DataFrame | Lazy evaluation, Transformation vs Action |
| 2 | Spark SQL & Catalyst | Execution plan, Predicate Pushdown |
| 3 | Shuffle & Partitioning | repartition, coalesce, Skew handling |
| 4 | Join Strategies | BHJ, SMJ, SHJ, Salting |
| 5 | Memory & Tuning | Unified Memory, Spill, OOM |
| 6 | Structured Streaming | Watermark, Window, Monitoring |
| 7 | Data Pipeline | Bronze/Silver/Gold, ETL, Quality checks |

In [18]:
spark.stop()
print('SparkSession stopped')
print('\n🎉 All 7 steps complete!')

SparkSession stopped

🎉 All 7 steps complete!
